In [27]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [28]:
df=pd.read_csv(r"C:\Users\kirti\Desktop\AI Engineer\End-To-End Project\Movie Recommendation System\data\tmdb_5000_movies.csv")

In [29]:
df.iloc[344]

budget                                                          100000000
genres                  [{"id": 28, "name": "Action"}, {"id": 53, "nam...
homepage                                                              NaN
id                                                                  44048
keywords                        [{"id": 163228, "name": "runaway train"}]
original_language                                                      en
original_title                                                Unstoppable
overview                A runaway train, transporting deadly, toxic ch...
popularity                                                      37.698465
production_companies    [{"name": "Twentieth Century Fox Film Corporat...
production_countries    [{"iso_3166_1": "US", "name": "United States o...
release_date                                                   2010-11-04
revenue                                                         167805466
runtime                               

In [30]:
df.isnull().sum()

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
dtype: int64

In [31]:
## Droping missing values of overview feature
df.drop([2656, 4140, 4431],axis=0,inplace=True)


In [32]:
df["overview"].isnull().sum()

np.int64(0)

### Overview Feature Cleaning step
- Convert all text to lowercase to eliminate case sensitivity issues.
- Remove punctuation, special characters, and HTML tags using regex (e.g., keep only letters, numbers, spaces). [.,!?]
- Strip extra whitespaces, including leading/trailing spaces and multiple spaces between words.
- Tokenize into words or n-grams, then remove stopwords (common words like "the", "is") using libraries like NLTK.17
- Apply lemmatization or stemming to reduce words to base forms (e.g., "running" → "run").

In [33]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))

def cleaning_overview(string):
    ## Lower case string conversion
    string=string.lower()

    ## Special character removal
    sp_chars=[".",",","!","?" '!', '@', '#', '$', '%', '^', '&', '*', '(', ')', '-', '_', '+', '=', '{', '}', '[', ']', '|', '\\',':', ';', '"', '\'', '<', '>', ',', '.', '?', '/','~', '`']
    for chars in sp_chars:
        string=string.replace(chars,"")
        
    ## Extra spaces removal
    lst=np.array(string.split(" "))

    ## Stopwords removal
    for stword in stop_words:
        for fstrword in lst:
            if stword == fstrword:
                lst[np.where(lst==fstrword)]=''
    final_str=" ".join(lst)
    final_str=" ".join(final_str.split())

    ## Word Tokenizer
    tokens= word_tokenize(final_str)

    ## Stemming
    stemmer=PorterStemmer()
    stemmed_words=[stemmer.stem(words) for words in tokens]

    
    
    return " ".join(stemmed_words)

In [34]:
df["overview"]=df["overview"].apply(cleaning_overview)

In [35]:
data=df[["title","overview"]]
data=data.reset_index(drop=True)

In [36]:
data["overview"]


0       22nd centuri parapleg marin dispatch moon pand...
1       captain barbossa long believ dead come back li...
2       cryptic messag bond ’ s past send trail uncov ...
3       follow death district attorney harvey dent bat...
4       john carter warweari former militari captain w...
                              ...                        
4795    el mariachi want play guitar carri famili trad...
4796    newlyw coupl honeymoon upend arriv respect sister
4797    sign seal deliv introduc dedic quartet civil s...
4798    ambiti new york attorney sam sent shanghai ass...
4799    ever sinc second grade first saw et extraterre...
Name: overview, Length: 4800, dtype: object

In [37]:
# joblib.dump(data,"cleaned_data.pkl")

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=5000)
vectors=vectorizer.fit_transform(data["overview"])

- Tf-idf vectorizer create vocabulary of unique words from the corpus
- sorts in descending order of frequency count, chooses top 5000
- calculates score for each document for each word of 5000 and stores in a sparse matrix.

In [39]:
data["overview"].shape

(4800,)

In [50]:
vectors

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 116100 stored elements and shape (4800, 5000)>

In [41]:
#import joblib
# joblib.dump(vectorizer,"fitted_tfidf.pkl")

In [42]:
# joblib.dump(vectors,"movie_vectors.pkl")

In [43]:
vectors.shape

(4800, 5000)

In [44]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix=cosine_similarity(vectors)

In [45]:
similarity_matrix

array([[1.        , 0.        , 0.        , ..., 0.0397924 , 0.        ,
        0.        ],
       [0.        , 1.        , 0.02705949, ..., 0.0288422 , 0.        ,
        0.        ],
       [0.        , 0.02705949, 1.        , ..., 0.02031242, 0.        ,
        0.        ],
       ...,
       [0.0397924 , 0.0288422 , 0.02031242, ..., 1.        , 0.01148622,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.01148622, 1.        ,
        0.00682867],
       [0.        , 0.        , 0.        , ..., 0.        , 0.00682867,
        1.        ]], shape=(4800, 4800))

In [46]:
# joblib.dump(similarity_matrix,"similarity_matrix.pkl")

In [47]:
def recommend_function(string):
    data.dropna(inplace=True)
    clean_str=cleaning_overview(string) ## Cleaning function
    idx=data.shape[0]+1
    
    if idx<=data.shape[0]:
        print("If condition")
        data.drop(idx,axis=0,inplace=True)
    else:
        data.loc[idx]=[np.nan,clean_str]
        str_vec=vectorizer.fit_transform(data["overview"])  ## TF-IDF Vectorizer
        sim_matrix=cosine_similarity(str_vec)   ## Cosine Similarity
        recommend_data=pd.Series(sim_matrix[data.shape[0]-1],name="Similarity_score").sort_values(ascending=False)[1:7]
        indexes=recommend_data.index
        scores=recommend_data.values
        final_table=pd.DataFrame(data.iloc[indexes])
        final_table["Confidence Scores"]=scores
    return final_table

In [48]:
fin_table=recommend_function("space mission, black hole, space ship, event horizon")

In [49]:
fin_table

,title,overview,Confidence Scores
1709,Space Pirate Captain Harlock,space pirat captain harlock fearless crew face...,0.318913
2129,The Black Hole,explor craft uss palomino return earth fruitle...,0.318159
1914,Lifeforce,space shuttl mission investig halley comet bri...,0.263322
461,Lost in Space,prospect continu life earth year 2058 grim rob...,0.244658
3533,Galaxina,galaxina lifelik voluptu android assign overse...,0.234562
3328,The Hole,move new neighbourhood brother dane amp luca n...,0.225903
